In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
MIN_OOT_DATE = '2024-03-01'
MAX_OOT_DATE = '2024-04-30'

In [3]:
base = pd.read_csv('base_sample.csv')
cancel = pd.read_csv('service_cancel.csv')
features = pd.read_csv('features.csv')

### Датасет с клиентами и их закрытыми заказами

In [4]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1519284 entries, 0 to 1519283
Data columns (total 17 columns):
 #   Column                 Non-Null Count    Dtype 
---  ------                 --------------    ----- 
 0   demand                 1519284 non-null  int64 
 1   client                 1519284 non-null  int64 
 2   save_stamp             1519284 non-null  object
 3   close_date             1519284 non-null  object
 4   first_service          1519284 non-null  int64 
 5   second_service         1519284 non-null  int64 
 6   first_service_cancel   1519284 non-null  int64 
 7   second_service_cancel  1519284 non-null  int64 
 8   second_service_refund  1519284 non-null  int64 
 9   first_service_refund   1519284 non-null  int64 
 10  asked_order_sum        1519284 non-null  int64 
 11  fact_order_sum         1519284 non-null  int64 
 12  first_loyalty_flag     1519284 non-null  int64 
 13  second_loyalty_flag    1519284 non-null  int64 
 14  third_loyalty_flag     1519284 non

### Датасет с информацией об отмене ДУ

In [5]:
cancel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381248 entries, 0 to 381247
Data columns (total 3 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   client               381248 non-null  int64 
 1   reg_date             381248 non-null  object
 2   first_ref_canc_date  141820 non-null  object
dtypes: int64(1), object(2)
memory usage: 8.7+ MB


### Функция добавления флага отмены (deny_flag) к заказу, на котором произошла отмена

In [6]:
base['save_stamp'] = pd.to_datetime(base['save_stamp'], format='mixed', errors='coerce')
base['close_date'] = pd.to_datetime(base['close_date'], format='mixed', errors='coerce')
cancel['reg_date'] = pd.to_datetime(cancel['reg_date'], format='mixed', errors='coerce')
cancel['first_ref_canc_date'] = pd.to_datetime(cancel['first_ref_canc_date'], format='mixed', errors='coerce')

In [7]:
def add_deny_flag(base, cancel):
    df_with_deny = base.merge(cancel[['client', 'first_ref_canc_date']], on='client', how='left')
    first_demand = df_with_deny.groupby('client')['save_stamp'].min().reset_index()
    first_demand.rename(columns={'save_stamp': 'first_save_stamp'}, inplace=True)
    df_with_deny = df_with_deny.merge(first_demand, on='client', how='left')
    df_with_deny = df_with_deny[(df_with_deny['first_ref_canc_date'].isna()) | (df_with_deny['first_ref_canc_date'] >= df_with_deny['first_save_stamp'])]
    df_valid = df_with_deny[df_with_deny['first_ref_canc_date'].notna()].copy()
    df_valid['diff'] = df_valid['first_ref_canc_date'] - df_valid['save_stamp']
    df_filtered = df_valid[df_valid['diff'] >= pd.Timedelta(0)]
    idx = df_filtered.groupby('client')['diff'].idxmin()
    df_with_deny['deny_flag'] = 0
    df_with_deny.loc[idx, 'deny_flag'] = 1
    
    return df_with_deny

In [8]:
df_with_deny = add_deny_flag(base, cancel)
df_with_deny.shape, len(df_with_deny['client'].unique())

((1519239, 20), 381238)

In [9]:
filt_cl = cancel[(cancel['reg_date'] >= MIN_OOT_DATE) & (cancel['reg_date'] <= MAX_OOT_DATE)]
df_with_deny['oot'] = 0
df_with_deny.loc[df_with_deny['client'].isin(filt_cl['client']), 'oot'] = 1

### Функция разметки датасета

Интерпретация: target = 1 на заказе, предшествующем заказу, на котором произошла отмена.

Целевая: отмена ДУ на следующем заказе.

Логика условий фильтрации прописана в тексте ВКР.

In [10]:
def get_marked_df(df): 
    df = df.sort_values(by='demand').reset_index(drop=True)
    df['target'] = 0
    df_final = []
    for _, df_cl in df.groupby('client'):
        df_cl = df_cl.reset_index(drop=True)
        if len(df_cl) == 1:
            continue
        mask_sure = (df_cl['second_loyalty_flag'].values == 1) | (df_cl['deny_flag'].values == 1) | (df_cl['third_loyalty_flag'].values == 1)
        mask_maybe = (df_cl['second_service'].values == 0) & (df_cl['first_service'].values == 0) & (df_cl['first_loyalty_flag'].values == 0)
        mask = mask_sure | mask_maybe
        if np.any(mask):
            ind = np.argmax(mask)
            if ind == 0:
                continue
            final_df_cl = df_cl.iloc[:ind].copy()
            final_df_cl.at[ind - 1, 'target'] = 1
        else:
            final_df_cl = df_cl
        df_final.append(final_df_cl)
    return pd.concat(df_final, ignore_index=True)

In [11]:
%%time
marked_df_with_deny = get_marked_df(df_with_deny)

CPU times: total: 1min 17s
Wall time: 1min 17s


In [12]:
marked_df_with_deny.shape, len(marked_df_with_deny['client'].unique()), marked_df_with_deny['target'].mean()

((321442, 22), 139089, 0.2886461632269585)

### Датасет с входными признаками

In [13]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321442 entries, 0 to 321441
Data columns (total 60 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   demand                        321442 non-null  int64  
 1   ltv_last                      321442 non-null  float64
 2   ltv_avg                       321442 non-null  float64
 3   comis_to_debt_ratio_max       321442 non-null  float64
 4   comis_to_debt_ratio           321442 non-null  float64
 5   comis_to_debt_ratio_last      321442 non-null  float64
 6   comis_to_paym_ratio_max       321433 non-null  float64
 7   comis_to_paym_ratio           321442 non-null  float64
 8   comis_to_paym_ratio_last      321319 non-null  float64
 9   dop_second_service_ratio      321442 non-null  float64
 10  dop_first_service_ratio       321442 non-null  float64
 11  dop_ratio                     321442 non-null  float64
 12  dop_sum_ratio_max             321442 non-nul

In [14]:
features.isna().sum()

demand                               0
ltv_last                             0
ltv_avg                              0
comis_to_debt_ratio_max              0
comis_to_debt_ratio                  0
comis_to_debt_ratio_last             0
comis_to_paym_ratio_max              9
comis_to_paym_ratio                  0
comis_to_paym_ratio_last           123
dop_second_service_ratio             0
dop_first_service_ratio              0
dop_ratio                            0
dop_sum_ratio_max                    0
dop_sum_ratio_last                   0
avg_order_time                       0
avg_time_btw_order              139089
avg_order_sum_to_asked               0
fact_order_sum_to_asked_max          0
fact_order_sum_to_asked_last         0
pl_last                              0
avg_pl                               0
sms_num                              0
email_num                        71215
call_num                        120393
age                                  0
gender                   

In [15]:
# необходимо убрать признаки sms_num, email_num, call_num, app_paltform_name, т.к. в процессе EDA и анализа пропусков выяснилось, 
# что в БД данные записывались некорректно
features = features.drop(columns=['sms_num', 'email_num', 'call_num', 'app_platform_name'])

Остальные пропуски несут определенный смысл (подробно прописано в тексте ВКР).

### Объединение датасетов

In [16]:
full_df = marked_df_with_deny[['demand', 'target', 'oot']].merge(features, on='demand', how='inner')

In [17]:
full_df

,demand,target,oot,ltv_last,ltv_avg,comis_to_debt_ratio_max,comis_to_debt_ratio,comis_to_debt_ratio_last,comis_to_paym_ratio_max,comis_to_paym_ratio,comis_to_paym_ratio_last,dop_second_service_ratio,dop_first_service_ratio,dop_ratio,dop_sum_ratio_max,dop_sum_ratio_last,avg_order_time,avg_time_btw_order,avg_order_sum_to_asked,fact_order_sum_to_asked_max,fact_order_sum_to_asked_last,pl_last,avg_pl,age,gender,income,call_cnt_last,cnt_calls_all,appeal_last_em,avg_appeal_em,all_appeal_em,appeal_last_lk,avg_appeal_lk,all_appeal_lk,start_amount_inc_ratio_last,avg_start_amount_inc_ratio,amount_inc_ratio_last,avg_amount_inc_ratio,sum_ratio_upto5,sum_ratio_upto10,extens_ratio,period30_ratio,dops_inc_ratio_last,dops_inc_ratio,cnt_session_last,avg_cnt_session,cnt_all_session,avg_approve,dop_sum_ratio,ltv,days_after_order_time,order_number,delay_ratio,avg_fact_order_sum,time_btw_last_order,lifetime_start_last,lifetime_last,order_time_last_ratio
0,23809874,1,0,8144.914,8144.910,0.290,0.290,0.290,0.090,0.090,0.090,1.000,1.000,1.000,0.600,0.000,69.000,NaN,1.000,1.000,1.000,16.290,16.290,20.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.080,0.080,0.080,0.080,1.000,1.000,1.000,1.000,0.000,0.050,10,10.000,11,1.000,0.600,8144.914,69,1,0.000,4000.000,NaN,22,70.000,3.182
1,23810121,1,0,4923.762,4923.760,0.160,0.160,0.160,0.100,0.100,0.100,1.000,1.000,1.000,0.300,0.000,16.000,NaN,1.000,1.000,1.000,4.920,4.920,25.000,1,70000.000,0,0,0,0.000,0,0,0.000,0,0.110,0.110,0.110,0.110,0.000,1.000,0.000,1.000,0.000,0.030,8,8.000,9,1.000,0.300,4923.763,16,1,1.000,8000.000,NaN,16,17.000,1.062
2,23845577,0,0,3257.543,3257.540,0.290,0.290,0.290,0.140,0.140,0.140,1.000,1.000,1.000,0.800,0.000,5.000,NaN,1.000,1.000,1.000,15.200,15.200,39.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.060,0.060,0.060,0.060,1.000,1.000,0.000,1.000,0.000,0.050,2,2.000,4,1.000,0.800,3257.543,5,1,1.000,3000.000,NaN,16,6.000,0.375
3,24846191,0,0,5518.203,4387.870,0.190,0.190,0.160,0.110,0.110,0.090,1.000,1.000,1.000,0.800,0.300,14.000,12.000,1.000,1.000,1.000,9.660,12.430,39.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.160,0.110,0.160,0.110,0.500,1.000,0.000,0.500,0.050,0.020,4,3.000,9,1.000,0.436,8775.745,53,2,1.000,5500.000,25.000,26,24.000,0.923
4,30554758,1,0,3805.142,4193.630,0.200,0.200,0.200,0.110,0.110,0.110,1.000,1.000,1.000,0.800,0.480,12.000,58.000,1.000,1.000,1.000,10.650,11.840,39.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.100,0.110,0.100,0.110,0.670,1.000,0.000,0.330,0.050,0.020,3,3.000,19,1.000,0.450,12580.888,211,3,1.000,5333.333,150.000,16,9.000,0.562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321437,46765176,0,0,3193.600,3193.600,0.080,0.080,0.080,0.060,0.060,0.060,1.000,1.000,1.000,0.240,0.000,7.000,NaN,1.000,1.000,1.000,3.190,3.190,20.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.330,0.330,0.330,0.330,0.000,1.000,0.000,1.000,0.000,0.080,1,1.000,2,1.000,0.240,3193.600,7,1,1.000,10000.000,NaN,30,7.000,0.233
321438,48222616,0,0,4112.642,3653.120,0.110,0.110,0.130,0.080,0.080,0.090,1.000,1.000,1.000,0.240,0.240,6.000,13.000,1.000,1.000,1.000,4.110,3.650,21.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.330,0.330,0.330,0.330,0.000,1.000,0.000,0.500,0.080,0.040,1,1.000,7,1.000,0.240,7306.243,38,2,1.000,10000.000,26.000,21,5.000,0.238
321439,49464215,0,0,3772.963,3693.070,0.110,0.110,0.130,0.080,0.080,0.090,1.000,1.000,1.000,0.240,0.240,4.000,16.000,0.890,1.000,0.670,3.770,3.690,21.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.500,0.390,0.330,0.330,0.000,1.000,0.000,0.670,0.080,0.030,0,0.670,8,1.000,0.240,11079.205,62,3,1.000,10000.000,24.000,19,0.000,0.000
321440,49512219,0,0,3857.883,3734.270,0.120,0.120,0.130,0.090,0.090,0.090,1.000,1.000,1.000,0.240,0.240,3.000,12.000,0.920,1.000,1.000,3.860,3.730,21.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.330,0.380,0.330,0.330,0.000,1.000,0.000,0.750,0.080,0.030,0,0.500,14

### EDA входных признаков

In [18]:
def main_info(df):
    print('Размер', df.shape)
    print('\nТипы данных:')
    print(df.dtypes.value_counts())
    
    print('\nРаспределение целевого признака:')
    print(df['target'].value_counts(normalize=True))
    
    print('\nПропущенные значения (%):')
    missing = df.isna().mean().sort_values(ascending=False)
    print(missing[missing > 0])

In [19]:
main_info(full_df)

Размер (321442, 58)

Типы данных:
float64    43
int64      15
Name: count, dtype: int64

Распределение целевого признака:
target
0   0.711
1   0.289
Name: proportion, dtype: float64

Пропущенные значения (%):
avg_time_btw_order            0.433
time_btw_last_order           0.433
comis_to_paym_ratio_last      0.000
dops_inc_ratio_last           0.000
start_amount_inc_ratio_last   0.000
amount_inc_ratio_last         0.000
income                        0.000
dops_inc_ratio                0.000
avg_amount_inc_ratio          0.000
avg_start_amount_inc_ratio    0.000
comis_to_paym_ratio_max       0.000
ltv                           0.000
dtype: float64


In [20]:
full_df.to_csv('df_for_selecting_feats.csv', index=False)

In [21]:
full_df

,demand,target,oot,ltv_last,ltv_avg,comis_to_debt_ratio_max,comis_to_debt_ratio,comis_to_debt_ratio_last,comis_to_paym_ratio_max,comis_to_paym_ratio,comis_to_paym_ratio_last,dop_second_service_ratio,dop_first_service_ratio,dop_ratio,dop_sum_ratio_max,dop_sum_ratio_last,avg_order_time,avg_time_btw_order,avg_order_sum_to_asked,fact_order_sum_to_asked_max,fact_order_sum_to_asked_last,pl_last,avg_pl,age,gender,income,call_cnt_last,cnt_calls_all,appeal_last_em,avg_appeal_em,all_appeal_em,appeal_last_lk,avg_appeal_lk,all_appeal_lk,start_amount_inc_ratio_last,avg_start_amount_inc_ratio,amount_inc_ratio_last,avg_amount_inc_ratio,sum_ratio_upto5,sum_ratio_upto10,extens_ratio,period30_ratio,dops_inc_ratio_last,dops_inc_ratio,cnt_session_last,avg_cnt_session,cnt_all_session,avg_approve,dop_sum_ratio,ltv,days_after_order_time,order_number,delay_ratio,avg_fact_order_sum,time_btw_last_order,lifetime_start_last,lifetime_last,order_time_last_ratio
0,23809874,1,0,8144.914,8144.910,0.290,0.290,0.290,0.090,0.090,0.090,1.000,1.000,1.000,0.600,0.000,69.000,NaN,1.000,1.000,1.000,16.290,16.290,20.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.080,0.080,0.080,0.080,1.000,1.000,1.000,1.000,0.000,0.050,10,10.000,11,1.000,0.600,8144.914,69,1,0.000,4000.000,NaN,22,70.000,3.182
1,23810121,1,0,4923.762,4923.760,0.160,0.160,0.160,0.100,0.100,0.100,1.000,1.000,1.000,0.300,0.000,16.000,NaN,1.000,1.000,1.000,4.920,4.920,25.000,1,70000.000,0,0,0,0.000,0,0,0.000,0,0.110,0.110,0.110,0.110,0.000,1.000,0.000,1.000,0.000,0.030,8,8.000,9,1.000,0.300,4923.763,16,1,1.000,8000.000,NaN,16,17.000,1.062
2,23845577,0,0,3257.543,3257.540,0.290,0.290,0.290,0.140,0.140,0.140,1.000,1.000,1.000,0.800,0.000,5.000,NaN,1.000,1.000,1.000,15.200,15.200,39.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.060,0.060,0.060,0.060,1.000,1.000,0.000,1.000,0.000,0.050,2,2.000,4,1.000,0.800,3257.543,5,1,1.000,3000.000,NaN,16,6.000,0.375
3,24846191,0,0,5518.203,4387.870,0.190,0.190,0.160,0.110,0.110,0.090,1.000,1.000,1.000,0.800,0.300,14.000,12.000,1.000,1.000,1.000,9.660,12.430,39.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.160,0.110,0.160,0.110,0.500,1.000,0.000,0.500,0.050,0.020,4,3.000,9,1.000,0.436,8775.745,53,2,1.000,5500.000,25.000,26,24.000,0.923
4,30554758,1,0,3805.142,4193.630,0.200,0.200,0.200,0.110,0.110,0.110,1.000,1.000,1.000,0.800,0.480,12.000,58.000,1.000,1.000,1.000,10.650,11.840,39.000,1,50000.000,0,0,0,0.000,0,0,0.000,0,0.100,0.110,0.100,0.110,0.670,1.000,0.000,0.330,0.050,0.020,3,3.000,19,1.000,0.450,12580.888,211,3,1.000,5333.333,150.000,16,9.000,0.562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321437,46765176,0,0,3193.600,3193.600,0.080,0.080,0.080,0.060,0.060,0.060,1.000,1.000,1.000,0.240,0.000,7.000,NaN,1.000,1.000,1.000,3.190,3.190,20.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.330,0.330,0.330,0.330,0.000,1.000,0.000,1.000,0.000,0.080,1,1.000,2,1.000,0.240,3193.600,7,1,1.000,10000.000,NaN,30,7.000,0.233
321438,48222616,0,0,4112.642,3653.120,0.110,0.110,0.130,0.080,0.080,0.090,1.000,1.000,1.000,0.240,0.240,6.000,13.000,1.000,1.000,1.000,4.110,3.650,21.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.330,0.330,0.330,0.330,0.000,1.000,0.000,0.500,0.080,0.040,1,1.000,7,1.000,0.240,7306.243,38,2,1.000,10000.000,26.000,21,5.000,0.238
321439,49464215,0,0,3772.963,3693.070,0.110,0.110,0.130,0.080,0.080,0.090,1.000,1.000,1.000,0.240,0.240,4.000,16.000,0.890,1.000,0.670,3.770,3.690,21.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.500,0.390,0.330,0.330,0.000,1.000,0.000,0.670,0.080,0.030,0,0.670,8,1.000,0.240,11079.205,62,3,1.000,10000.000,24.000,19,0.000,0.000
321440,49512219,0,0,3857.883,3734.270,0.120,0.120,0.130,0.090,0.090,0.090,1.000,1.000,1.000,0.240,0.240,3.000,12.000,0.920,1.000,1.000,3.860,3.730,21.000,1,30000.000,0,0,0,0.000,0,0,0.000,0,0.330,0.380,0.330,0.330,0.000,1.000,0.000,0.750,0.080,0.030,0,0.500,14